<a href="https://colab.research.google.com/github/MONISHKA1607/Accelerated_Data_Science_UCS547/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


1. You are given a large NumPy array of size 5000000 ini alized with random
values. Compute the following element-wise opera on: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64



In [1]:
!pip install numba

In [2]:
import numpy as np
from numba import cuda
import time

In [15]:
N = 5_000_000
arr = np.random.rand(N)

def cpu_compute(x):
    return x*x + 3*x + 5

start = time.time()
cpu_result = cpu_compute(arr)
cpu_time = time.time() - start

print("CPU Time:", cpu_time)

CPU Time: 0.03989005088806152


In [9]:
@cuda.jit
def gpu_compute_float32(x, out):
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i]*x[i] + 3*x[i] + 5

In [16]:
arr32 = arr.astype(np.float32)
out32 = np.zeros_like(arr32)

d_x = cuda.to_device(arr32)
d_out = cuda.device_array_like(arr32)

threads = 256
blocks = (N + threads - 1) // threads

gpu_compute_float32[blocks, threads](d_x, d_out)
cuda.synchronize()

start = time.time()
gpu_compute_float32[blocks, threads](d_x, d_out)
cuda.synchronize()
gpu_time32 = time.time() - start

d_out.copy_to_host(out32)

print("GPU float32 Kernel Time:", gpu_time32)

GPU float32 Kernel Time: 0.0008311271667480469


In [17]:
@cuda.jit
def gpu_compute_float64(x, out):
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i]*x[i] + 3*x[i] + 5

In [18]:
arr64 = arr.astype(np.float64)
out64 = np.zeros_like(arr64)

d_x64 = cuda.to_device(arr64)
d_out64 = cuda.device_array_like(arr64)

gpu_compute_float64[blocks, threads](d_x64, d_out64)
cuda.synchronize()

start = time.time()
gpu_compute_float64[blocks, threads](d_x64, d_out64)
cuda.synchronize()
gpu_time64 = time.time() - start

d_out64.copy_to_host(out64)

print("GPU float64 Kernel Time:", gpu_time64)

GPU float64 Kernel Time: 0.0005834102630615234


2. Implement and benchmark a 1-D histogram computa on for 1 million
random values in Python using Numba. Compare different approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness.
(Ref:
h ps://numba.pydata.org/numba
examples/examples/density_esma on/histogram/results.html )

In [21]:
import numpy as np
import time
from numba import njit

data = np.random.randint(0, 100, size=1_000_000)
bins = 100

In [22]:
def py_hist(data, bins):
    hist = [0]*bins
    for x in data:
        hist[x] += 1
    return hist

start = time.time()
h_py = py_hist(data, bins)
py_time = time.time() - start

print("Python Time:", py_time)

Python Time: 0.26584839820861816


In [23]:
start = time.time()
h_np, _ = np.histogram(data, bins=bins, range=(0, bins))
np_time = time.time() - start

print("NumPy Time:", np_time)

NumPy Time: 0.03197026252746582


In [24]:
@njit
def numba_hist(data, bins):
    hist = np.zeros(bins, dtype=np.int32)
    for i in range(data.size):
        hist[data[i]] += 1
    return hist

numba_hist(data, bins)

start = time.time()
h_nb = numba_hist(data, bins)
nb_time = time.time() - start

print("Numba Time:", nb_time)

Numba Time: 0.001239776611328125


3. Write a func on monte_carlo_pi(nsamples) that esmates the value of π by
genera ng random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).
a. Implement the func on in pure Python first and later create a Numba
version.  
b. Program a script to compare the execu on me for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).  
c. Why does the very first execu on of the Numba func on take slightly
longer than the second execu on?

In [26]:
import random
import time

def monte_carlo_py(n):
    inside = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / n

start = time.time()
pi_py = monte_carlo_py(5_000_000)
py_time = time.time() - start

print("Python PI:", pi_py)
print("Python Time:", py_time)

Python PI: 3.1423448
Python Time: 1.3356661796569824


In [27]:
from numba import njit
import numpy as np

@njit
def monte_carlo_numba(n):
    inside = 0
    for i in range(n):
        x = np.random.random()
        y = np.random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / n

monte_carlo_numba(10)

start = time.time()
pi_nb = monte_carlo_numba(5_000_000)
nb_time = time.time() - start

print("Numba PI:", pi_nb)
print("Numba Time:", nb_time)
print("Speedup:", py_time / nb_time)

Numba PI: 3.1408544
Numba Time: 0.057135820388793945
Speedup: 23.377036867032487


4. You have a 1D NumPy array represen ng pixel intensi es (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.
a. Write a func on adjust_brightness(pixel_value) using the @vectorize
decorator.
b. Apply this func on to an array of 10 million random integers.
c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the me difference when the work is automa cally split
across your CPU cores.
d. What happens if you try to pass a list instead of a NumPy array to this
func on?

In [28]:
import numpy as np
import time
from numba import vectorize

In [29]:
@vectorize(['int64(int64)'])
def adjust_brightness(x):
    val = int(x * 1.2)
    return 255 if val > 255 else val

In [30]:
pixels = np.random.randint(0, 256, size=10_000_000)

start = time.time()
result = adjust_brightness(pixels)
t1 = time.time() - start

print("Vectorized Time:", t1)

Vectorized Time: 0.026411771774291992


In [40]:
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(x):
    val = int(x * 1.2)
    return 255 if val > 255 else val

start = time.time()
result_parallel = adjust_brightness_parallel(pixels)
t2 = time.time() - start

print("Parallel Time:", t2)

Parallel Time: 0.02124762535095215


5. Write Python code to generate synthe c training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logis c regression
using the mathema cal formula for gradient descent:
a. Using standard NumPy (without Numba)
b. Using Numba JIT accelera on
c. Compare correctness and performance.

In [41]:
import numpy as np
import time

X = np.random.randn(100000, 10)
y = np.random.choice([0, 1], size=100000)

In [42]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_numpy(X, y, lr=0.01, epochs=50):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        z = X @ w
        pred = sigmoid(z)
        grad = X.T @ (pred - y) / len(y)
        w -= lr * grad
    return w

start = time.time()
w_np = train_numpy(X, y)
np_time = time.time() - start

print("NumPy Time:", np_time)

NumPy Time: 0.3193936347961426


In [43]:
from numba import njit

@njit
def train_numba(X, y, lr, epochs):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        z = X @ w
        pred = 1 / (1 + np.exp(-z))
        grad = X.T @ (pred - y) / len(y)
        w -= lr * grad
    return w

# Warm-up
train_numba(X, y, 0.01, 1)

start = time.time()
w_nb = train_numba(X, y, 0.01, 50)
nb_time = time.time() - start

print("Numba Time:", nb_time)

Numba Time: 0.17628812789916992


6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [45]:
import numpy as np
from numba import cuda

In [46]:
@cuda.jit
def mat_add(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

In [47]:
N = 1024

A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
C = np.zeros((N, N), dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(A)

threads = (16, 16)
blocks = ((N + 15)//16, (N + 15)//16)

# Warm-up
mat_add[blocks, threads](d_A, d_B, d_C)
cuda.synchronize()

# Timing kernel only
import time
start = time.time()
mat_add[blocks, threads](d_A, d_B, d_C)
cuda.synchronize()
gpu_time = time.time() - start

d_C.copy_to_host(C)

print("GPU Time:", gpu_time)

GPU Time: 0.0003199577331542969
